{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "e0e44290",
   "metadata": {},
   "source": [
    "# Projeto Seq2Seq para Geração de Descrições de Imagem\n",
    "\n",
    "Este notebook documenta o comportamento de um projeto que utiliza uma arquitetura seq2seq com atenção para gerar descrições de imagens a partir de um conjunto de labels. O fluxo do projeto é composto pelas seguintes etapas:\n",
    "\n",
    "1. **Geração de Dados:** Um script gera exemplos sintéticos (e também incorpora exemplos reais) utilizando templates variados para produzir descrições e um conjunto de labels correspondentes.\n",
    "2. **Manipulação Humana:** Entre a geração e o pré-processamento, há uma intervenção manual para garantir que o dataset seja consistente, removendo duplicados e corrigindo possíveis inconsistências.\n",
    "3. **Pré-processamento:** Limpeza dos textos, normalização, tokenização dos labels e descrições e divisão em conjuntos de treino e teste.\n",
    "4. **Treino do Modelo:** Treinamento de um modelo seq2seq com atenção utilizando PyTorch, onde o encoder processa os labels e o decoder gera a descrição.\n",
    "5. **Inferência:** Uso do modelo treinado para gerar uma descrição dada uma lista de labels.\n",
    "\n",
    "A seguir, serão apresentados trechos de código e explicações referentes a cada etapa do projeto."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e59a4fce",
   "metadata": {},
   "source": [
    "## 1. Geração de Dados\n",
    "\n",
    "O script `1_generate.py` cria um dataset sintético a partir de um conjunto de categorias (como veículos, pessoas, infraestrutura, etc.) e condições contextuais (tempo, localização, comportamento). Para cada exemplo, uma descrição é gerada utilizando um template em português e os labels associados são extraídos. Ao final, os dados são salvos em um arquivo JSON.\n",
    "\n",
    "Exemplo de trecho de código (simplificado):"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "1a8d7e3e",
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "import random\n",
    "\n",
    "# Definição de categorias e templates\n",
    "categories = {\n",
    "    \"veiculos\": [\"carro\", \"autocarro\", \"camião\"],\n",
    "    \"pessoas\": [\"peão\", \"ciclista\"],\n",
    "    \"infraestrutura\": [\"passadeira\", \"semáforo\"],\n",
    "    \"sinais_transito\": [\"sinal de stop\", \"sinal de limite de velocidade\", \"sinal de passadeira\"]\n",
    "}\n",
    "\n",
    "weather_conditions = [\"céu limpo\", \"nublado\", \"chuvoso\", \"tempestuoso\", \"com nevoeiro\", \"vento forte\", \"com neve\"]\n",
    "time_of_day = [\"amanhecer\", \"anoitecer\", \"dia\", \"noite\"]\n",
    "locations = [\"residencial\", \"parque de estacionamento\", \"túnel\", \"cidade\", \"autoestrada\"]\n",
    "object_behaviors = [\"a circular rapidamente\", \"parados no semáforo\", \"a aguardar para atravessar\", \"estacionados\", \"a circular lentamente\", \"a atravessar a via\", \"em obras\"]\n",
    "\n",
    "# Seleção aleatória dos itens e geração de exemplos\n",
    "def generate_synthetic_data(num_samples=1000):\n",
    "    synthetic_data = []\n",
    "    for _ in range(num_samples):\n",
    "        veiculos = random.sample(categories[\"veiculos\"], k=random.randint(1,2))\n",
    "        pessoas = random.sample(categories[\"pessoas\"], k=random.randint(1,2))\n",
    "        infraestrutura = random.sample(categories[\"infraestrutura\"], k=random.randint(1,2))\n",
    "        sinais = random.sample(categories[\"sinais_transito\"], k=random.randint(1,2))\n",
    "        \n",
    "        weather = random.choice(weather_conditions)\n",
    "        time_of_day_choice = random.choice(time_of_day)\n",
    "        location = random.choice(locations)\n",
    "        traffic = random.choice([\"trânsito leve\", \"trânsito moderado\", \"trânsito intenso\", \"engarrafamento\"])\n",
    "        behavior = random.choice(object_behaviors)\n",
    "        pessoas_behavior = random.choice([\"a aguardar para atravessar\", \"a caminhar pela via\", \"à espera junto ao semáforo\"])\n",
    "\n",
    "        # Seleção de um template e formatação da descrição\n",
    "        templates = [\n",
    "            \"{time}, num dia {weather}, observam-se {veiculos} {behavior}, enquanto {pessoas} estão {pessoas_behavior}, {location}, com {traffic}.\",\n",
    "            \"Durante a {time}, sob um clima {weather}, podem ver-se {pessoas} {pessoas_behavior}, bem como {infraestrutura} e {sinais} visíveis {location}, além de {veiculos} {behavior}, com {traffic}.\"\n",
    "        ]\n",
    "        template = random.choice(templates)\n",
    "        description = template.format(\n",
    "            veiculos=\", \".join(veiculos),\n",
    "            pessoas=\", \".join(pessoas),\n",
    "            infraestrutura=\", \".join(infraestrutura),\n",
    "            sinais=\", \".join(sinais),\n",
    "            weather=weather,\n",
    "            time=time_of_day_choice,\n",
    "            location=location,\n",
    "            traffic=traffic,\n",
    "            behavior=behavior,\n",
    "            pessoas_behavior=pessoas_behavior\n",
    "        )\n",
    "        \n",
    "        # Adicionar tokens de início e fim\n",
    "        description = f\"startseq {description} endseq\"\n",
    "        \n",
    "        example = {\"labels\": veiculos + pessoas + infraestrutura + sinais, \"description\": description, \"origem\": \"sintético\"}\n",
    "        synthetic_data.append(example)\n",
    "    return synthetic_data\n",
    "\n",
    "# Gerar e salvar os dados\n",
    "data = generate_synthetic_data(1000)\n",
    "with open('data/synthetic_and_real_data.json', 'w', encoding='utf-8') as f:\n",
    "    json.dump(data, f, indent=4, ensure_ascii=False)\n",
    "print(\"Dados sintéticos salvos com sucesso!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d186a1a9",
   "metadata": {},
   "source": [
    "## 2. Manipulação e Pré-processamento dos Dados\n",
    "\n",
    "Após a geração dos dados, houve uma etapa de verificação manual para garantir a qualidade do dataset. Essa etapa inclui a remoção de duplicados e a verificação de consistência, como evidenciado pelo script `deteta_duplicados.py`.\n",
    "\n",
    "Em seguida, o script `2_pre_processamento.py` realiza a limpeza dos textos (remoção de acentos, conversão para minúsculas e remoção de caracteres especiais), tokeniza os textos (labels e descrições) utilizando uma implementação customizada (`SimpleTokenizer`) e aplica padding nas sequências. Por fim, os dados são divididos em conjuntos de treino e teste e salvos em arquivos para uso futuro."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9b327453",
   "metadata": {},
   "outputs": [],
   "source": [
    "from tokenizer_utils import SimpleTokenizer, pad_sequences\n",
    "import json\n",
    "import re\n",
    "import unicodedata\n",
    "import numpy as np\n",
    "from sklearn.model_selection import train_test_split\n",
    "\n",
    "# Função para limpar o texto\n",
    "def clean_text(text):\n",
    "    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8', 'ignore')\n",
    "    text = text.lower()\n",
    "    text = re.sub(r'[^a-z0-9\\s]', '', text)\n",
    "    text = re.sub(r'\\s+', ' ', text).strip()\n",
    "    return text\n",
    "\n",
    "# Carregar dataset (exemplo real/sintético)\n",
    "with open('data/exemplo_dataset.json', 'r', encoding='utf-8') as f:\n",
    "    data = json.load(f)\n",
    "\n",
    "# Processar descrições e labels\n",
    "descriptions = []\n",
    "labels_text = []\n",
    "for d in data:\n",
    "    desc = d.get(\"description\", \"\")\n",
    "    desc = desc.replace(\"startseq\", \"\").replace(\"endseq\", \"\").strip()\n",
    "    desc_clean = \"startseq \" + clean_text(desc) + \" endseq\"\n",
    "    descriptions.append(desc_clean)\n",
    "    labs = d.get(\"labels\", [])\n",
    "    labels_text.append(\" \".join([clean_text(l) for l in labs]))\n",
    "\n",
    "# Criar tokenizadores\n",
    "label_tokenizer = SimpleTokenizer(oov_token=\"<UNK>\", filters='')\n",
    "label_tokenizer.fit_on_texts(labels_text)\n",
    "\n",
    "description_tokenizer = SimpleTokenizer(oov_token=\"<UNK>\", filters='')\n",
    "description_tokenizer.fit_on_texts(descriptions)\n",
    "\n",
    "# Converter textos em sequências\n",
    "label_seq = label_tokenizer.texts_to_sequences(labels_text)\n",
    "desc_seq = description_tokenizer.texts_to_sequences(descriptions)\n",
    "\n",
    "# Aplicar padding\n",
    "max_label_length = min(max(len(seq) for seq in label_seq), 30)\n",
    "max_desc_length = min(max(len(seq) for seq in desc_seq), 270)\n",
    "label_padded = pad_sequences(label_seq, maxlen=max_label_length, padding='post')\n",
    "desc_padded = pad_sequences(desc_seq, maxlen=max_desc_length, padding='post')\n",
    "\n",
    "# Dividir os dados\n",
    "label_train, label_test, desc_train, desc_test = train_test_split(label_padded, desc_padded, test_size=0.2, random_state=42)\n",
    "print(\"Pré-processamento concluído.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4f18bba1",
   "metadata": {},
   "source": [
    "## 3. Treinamento do Modelo Seq2Seq com Atenção\n",
    "\n",
    "O script `3_train.py` implementa o treinamento de um modelo seq2seq com atenção (baseado na abordagem Luong). O modelo consiste de um encoder que processa os labels e um decoder que gera a descrição. Durante o treinamento, é utilizado o teacher forcing e a função de perda é a entropia cruzada (com ignorância do índice de padding).\n",
    "\n",
    "Segue um resumo do pipeline de treinamento:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "ae6950cf",
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.optim as optim\n",
    "from torch.utils.data import Dataset, DataLoader, random_split\n",
    "from tqdm import tqdm\n",
    "\n",
    "# Exemplo simplificado de definição do dataset\n",
    "class Seq2SeqDataset(Dataset):\n",
    "    def __init__(self, encoder_data, decoder_input, decoder_target):\n",
    "        self.encoder_data = encoder_data\n",
    "        self.decoder_input = decoder_input\n",
    "        self.decoder_target = decoder_target\n",
    "    def __len__(self):\n",
    "        return len(self.encoder_data)\n",
    "    def __getitem__(self, idx):\n",
    "        return (\n",
    "            torch.LongTensor(self.encoder_data[idx]),\n",
    "            torch.LongTensor(self.decoder_input[idx]),\n",
    "            torch.LongTensor(self.decoder_target[idx])\n",
    "        )\n",
    "\n",
    "# Definição do modelo (encoder, decoder com atenção e seq2seq) é feita conforme o script\n",
    "# e o treinamento é realizado iterando sobre o DataLoader e salvando o melhor modelo.\n",
    "\n",
    "print(\"Treinamento do modelo (exemplo simplificado) iniciado...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d6eac35a",
   "metadata": {},
   "source": [
    "## 4. Inferência\n",
    "\n",
    "No script `4_inference.py`, o modelo treinado é carregado e usado para gerar uma descrição a partir de um conjunto de labels fornecido. O processo envolve a tokenização dos labels, passagem pelo encoder e, em seguida, geração da sequência de saída pelo decoder até encontrar o token de fim (`endseq`).\n",
    "\n",
    "Segue um exemplo de função de inferência:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "5f0f77f6",
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import pickle\n",
    "from train import Encoder, Decoder, Seq2Seq\n",
    "\n",
    "# Carregar tokenizadores e modelo\n",
    "with open('models/label_tokenizer.pkl', 'rb') as f:\n",
    "    label_tokenizer = pickle.load(f)\n",
    "with open('models/description_tokenizer.pkl', 'rb') as f:\n",
    "    description_tokenizer = pickle.load(f)\n",
    "\n",
    "label_vocab_size = len(label_tokenizer.word_index) + 1\n",
    "desc_vocab_size = len(description_tokenizer.word_index) + 1\n",
    "\n",
    "device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n",
    "\n",
    "encoder = Encoder(label_vocab_size, emb_dim=300, hidden_dim=256).to(device)\n",
    "decoder = Decoder(desc_vocab_size, emb_dim=300, enc_hidden_dim=256, dec_hidden_dim=512).to(device)\n",
    "model = Seq2Seq(encoder, decoder).to(device)\n",
    "model.load_state_dict(torch.load('models/best_model.pt', map_location=device))\n",
    "model.eval()\n",
    "\n",
    "def generate_description(labels):\n",
    "    # Preprocessamento dos labels\n",
    "    label_str = \" \".join(labels).lower()\n",
    "    label_seq = label_tokenizer.texts_to_sequences([label_str])\n",
    "    label_seq = torch.LongTensor(label_seq).to(device)\n",
    "\n",
    "    with torch.no_grad():\n",
    "        encoder_outputs, hidden, cell = model.encoder(label_seq)\n",
    "        input_token = torch.LongTensor([[description_tokenizer.word_index['startseq']]]).to(device)\n",
    "        output_sentence = []\n",
    "        for _ in range(270):\n",
    "            output, hidden, cell = model.decoder(input_token.squeeze(1), hidden, cell, encoder_outputs)\n",
    "            top1 = output.argmax(-1).item()\n",
    "            if top1 == description_tokenizer.word_index.get('endseq'):\n",
    "                break\n",
    "            word = description_tokenizer.index_word.get(top1, '')\n",
    "            output_sentence.append(word)\n",
    "            input_token = torch.LongTensor([[top1]]).to(device)\n",
    "    return \" \".join(output_sentence)\n",
    "\n",
    "# Exemplo de uso\n",
    "example_labels = [\"céu_limpo\", \"noite\", \"parque_de_estacionamento\", \"sinal_de_stop\", \"carro\", \"peao\", \"passadeira\"]\n",
    "print(\"Labels:\", example_labels)\n",
    "print(\"Descrição gerada:\", generate_description(example_labels))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "04f7d1cf",
   "metadata": {},
   "source": [
    "## Considerações Finais\n",
    "\n",
    "Este projeto exemplifica a integração de várias etapas essenciais para o desenvolvimento de um sistema de geração de descrições a partir de labels:\n",
    "\n",
    "- **Geração de Dados:** Uso de templates e dados sintéticos para criar um grande dataset.\n",
    "- **Intervenção Humana:** Verificação e remoção de duplicados (e possivelmente outros ajustes) para garantir a qualidade dos dados.\n",
    "- **Pré-processamento:** Limpeza, tokenização e preparação dos dados para o treinamento.\n",
    "- **Modelagem:** Implementação de uma arquitetura seq2seq com atenção para gerar descrições de forma condicional.\n",
    "- **Inferência:** Geração de descrições a partir de novas entradas de labels.\n",
    "\n",
    "Este notebook serve tanto como documentação quanto como guia prático para execução e experimentação com o projeto."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.x"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
